# At the end of this notebook you'll be able to analyze:

On-Time Performance (OTP)
Delay Rate %
Cancellation Rate %
Delay Minutes
Delay Reasons
Cancellation Reasons
Weather Impact
Security Impact
Route Reliability
Pre/Post October 7 analysis
Airport congestion trends
Route operational quality

In [23]:
import pandas as pd
import numpy as np

np.random.seed(42)

OUTPUT_FOLDER = "../Data/Generated/"

In [24]:
# Loading Operations data
fact = pd.read_csv(
    OUTPUT_FOLDER + "fact_operations_stage1.csv"
)

print(fact.shape)

(56992, 43)


In [25]:
# Baseline event probablity: 18% delay, 2% cancellation, 80% on-time
BASE_DELAY_RATE = 0.18
BASE_CANCEL_RATE = 0.02

In [26]:
#Baseline operational risk index
fact["Operational_Risk"] = 1.0

In [27]:
#Higher security risk increases delays and cancellations.
fact["Operational_Risk"] *= (
    1 +
    fact["Security_Risk_Index"] * 0.08
)

In [28]:
# Winter months have higher operational risk due to weather conditions.
winter_months = [12,1,2]

fact.loc[
    fact["Month"].isin(winter_months),
    "Operational_Risk"
] *= 1.15

In [29]:
# Summer months = Crowded airports = higher operational risk
fact.loc[
    fact["Month"].isin([7,8]),
    "Operational_Risk"
] *= 1.10

In [30]:
# longer route experience more delays
fact.loc[
    fact["Distance_KM"] > 2500,
    "Operational_Risk"
] *= 1.10

In [31]:
# Delay Probability
fact["Delay_Probability"] = (
    BASE_DELAY_RATE
    *
    fact["Operational_Risk"]
)

fact["Delay_Probability"] = (
    fact["Delay_Probability"]
    .clip(upper=0.60)
)

In [32]:
# Cancellation Probability
fact["Cancel_Probability"] = (
    BASE_CANCEL_RATE
    *
    fact["Operational_Risk"]
)

fact["Cancel_Probability"] = (
    fact["Cancel_Probability"]
    .clip(upper=0.15)
)

In [33]:
# Generating Flight Status
rand_cancel = np.random.rand(len(fact))
rand_delay = np.random.rand(len(fact))

fact["Flight_Status"] = "On Time"

fact.loc[
    rand_delay < fact["Delay_Probability"],
    "Flight_Status"
] = "Delayed"

fact.loc[
    rand_cancel < fact["Cancel_Probability"],
    "Flight_Status"
] = "Cancelled"

In [34]:
fact["Delay_Minutes"] = 0

In [35]:
# Only delayed flights get delay minutes
delay_mask = (
    fact["Flight_Status"] == "Delayed"
)

fact.loc[
    delay_mask,
    "Delay_Minutes"
] = np.random.gamma(
    shape=2,
    scale=25,
    size=delay_mask.sum()
).round()

In [36]:
# Delay reasons
delay_reasons = [
    "Weather",
    "Technical",
    "Crew",
    "Airport Congestion",
    "ATC",
    "Security"
]

In [37]:
fact["Delay_Reason"] = None

delay_mask = (
    fact["Flight_Status"] == "Delayed"
)

fact.loc[
    delay_mask &
    (fact["Security_Risk_Index"] >= 3),
    "Delay_Reason"
] = np.random.choice(
    ["Security","ATC","Airport Congestion"],
    size=(
        delay_mask &
        (fact["Security_Risk_Index"] >= 3)
    ).sum(),
    p=[0.5,0.25,0.25]
)

remaining = (
    delay_mask &
    fact["Delay_Reason"].isna()
)

fact.loc[
    remaining,
    "Delay_Reason"
] = np.random.choice(
    [
        "Weather",
        "Technical",
        "Crew",
        "Airport Congestion",
        "ATC"
    ],
    size=remaining.sum(),
    p=[0.20,0.20,0.10,0.25,0.25]
)

In [38]:
# Cancellation Reasons
cancel_reasons = [
    "Weather",
    "Security",
    "Technical",
    "Operational"
]

In [39]:
fact["Cancellation_Reason"] = None

cancel_mask = (
    fact["Flight_Status"] == "Cancelled"
)

fact.loc[
    cancel_mask &
    (fact["Security_Risk_Index"] >= 3),
    "Cancellation_Reason"
] = np.random.choice(
    [
        "Security",
        "Operational"
    ],
    size=(
        cancel_mask &
        (fact["Security_Risk_Index"] >= 3)
    ).sum(),
    p=[0.70,0.30]
)

remaining = (
    cancel_mask &
    fact["Cancellation_Reason"].isna()
)

fact.loc[
    remaining,
    "Cancellation_Reason"
] = np.random.choice(
    [
        "Weather",
        "Technical",
        "Operational"
    ],
    size=remaining.sum(),
    p=[0.40,0.30,0.30]
)

In [40]:
# Relaiability KPIs
print(
    fact["Flight_Status"]
    .value_counts(normalize=True)
    * 100
)

Flight_Status
On Time      72.611946
Delayed      24.559587
Cancelled     2.828467
Name: proportion, dtype: float64


In [41]:
fact.loc[
    fact["Flight_Status"]=="Delayed",
    "Delay_Minutes"
].describe()

count    13997.000000
mean        50.477459
std         35.797452
min          0.000000
25%         25.000000
50%         42.000000
75%         68.000000
max        378.000000
Name: Delay_Minutes, dtype: float64

In [42]:
fact.to_csv(
    OUTPUT_FOLDER +
    "fact_operations_stage2.csv",
    index=False
)

print("fact_operations_stage2.csv saved")

fact_operations_stage2.csv saved
